# Human Oversight — Dev Log

## Objetivo e papel no pipeline

`core/human_oversight` é a fila real de itens que exigem revisão humana —
a peça que faltava para o "direito à revisão" do Art. 20 da LGPD virar um
fluxo operacional de verdade, não só uma flag (`PolicyDecisionStatus.REQUIRES_HUMAN_REVIEW`)
dentro de um `PolicyDecision` do V1.

Diferente do `audit_logs` (append-only, imutável), um item da fila de revisão
é **mutável**: nasce `pending` e vira `approved`/`rejected` uma única vez —
persistência em JSON com leitura+escrita completa, não `.jsonl`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.human_oversight.queue import OversightQueue
from shared.schemas import RiskLevel, OversightItemStatus

demo_dir = Path(tempfile.mkdtemp(prefix="human_oversight_demo_"))
queue = OversightQueue(storage_path=demo_dir / "queue.json")

item1 = queue.enqueue(
    subject="Triagem automatizada de currículos — Vaga Estágio Dados",
    reason="policy_engine retornou REQUIRES_HUMAN_REVIEW (POL-008: decisão automatizada de alto impacto)",
    risk_level=RiskLevel.MEDIUM,
)
item2 = queue.enqueue(
    subject="Score de crédito automatizado — Cliente PJ",
    reason="policy_engine retornou REQUIRES_HUMAN_REVIEW (dado sensível + decisão automatizada)",
    risk_level=RiskLevel.HIGH,
)
print(f"2 itens enfileirados: {item1.item_id[:8]}... e {item2.item_id[:8]}...")

decided = queue.decide(item1.item_id, approve=True, reviewer="ana.compliance@empresa.com", notes="Revisado, critérios objetivos e documentados.")
print(f"Item 1 decidido: status={decided.status.value}, revisor={decided.reviewer}")

pending = queue.list_items(status=None)
still_pending = queue.list_items(status=OversightItemStatus.PENDING)
print(f"Total de itens: {len(pending)} | ainda pendentes: {len(still_pending)}")
for p in still_pending:
    print(f"  - {p.subject} (risco={p.risk_level.value})")

2 itens enfileirados: 72368c48... e 51e29e79...
Item 1 decidido: status=approved, revisor=ana.compliance@empresa.com
Total de itens: 2 | ainda pendentes: 1
  - Score de crédito automatizado — Cliente PJ (risco=high)


## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/human_oversight/tests -v
```

10 testes contra arquivo temporário real: item criado `pending`;
persistência sobrevive a uma nova instância de `OversightQueue` sobre o
mesmo arquivo (prova de persistência real, não em memória); filtro por
status; decisão preenche campos corretos; decidir duas vezes levanta erro;
item/decisão desconhecidos levantam erro; fila vazia.

## Handoff Summary

- **Status:** ✅ done — 10/10 testes passando.
- **TODO onda futura:** `governance_copilot` enfileirando automaticamente a
  cada `PolicyDecision` com `REQUIRES_HUMAN_REVIEW`; autenticação real de
  quem decide (hoje `reviewer` é string livre — mesma decisão de escopo do
  V1: uso local).